# 03c · Ensemble matcher: LightGBM + XGBoost + CatBoost + neural network

The data, features and decision rule of `03_full_lightgbm_submission.ipynb`, with three more matchers next to
LightGBM and blends of them:

1. **Training pairs and features:** identical to 03: same buckets, same 85 features, and the same
   `TRAIN_S1_PER_COUNTRY` random train S1 per country and 3 folds (same seed). The LightGBM row below therefore
   reproduces 03's cross-validation score, and every model is compared on exactly the same held-out pairs.
2. **Cross-validation of every model** (grouped by S1): LightGBM (03's settings), XGBoost and CatBoost (GPU), and
   a small neural network (GPU).
3. **Blends**, scored on the same held-out predictions:
   * `mean`: the average of the models' probabilities
   * `stack`: a logistic regression on the models' log-odds, fitted on the other folds' held-out predictions (so
     its score is honest too)
4. **Choice:** the table shows each option's gain over LightGBM alone, with ± 2 standard errors (paired over S1).
   `ENSEMBLE=auto` takes the option with the best cross-validation score, but only if its gain is larger than its
   ± 2 SE; otherwise it keeps LightGBM. A gain within noise isn't worth giving up LightGBM's probabilities, which
   the France cutoff was tuned on.
5. **Country transfer** (the stand-in for France) for every model and blend.
6. **Final models** on all training pairs, then the **test prediction** (features computed once per chunk and
   scored by every model the choice uses), one owner, cutoff, and the same submission files and checks as 03.

**Input:** as 03. **Output** (`submission/`): as 03, plus:

| File | Content |
|---|---|
| `model/model_comparison.csv` | every model and blend: cutoff, cross-validation scores, gain over LightGBM |
| `model/oof_pairs.parquet`, `test_probabilities/<country>.parquet` | as in 03; `p` is the chosen option, and `p_lgbm`, `p_xgb`, `p_cat`, `p_nn` each model's probability |
| `model/lgbm_matcher.txt`, `xgb_matcher.json`, `cat_matcher.cbm`, `nn_matcher.pt` | the final models the choice uses |

04 rethreshold works on this output as on 03's. Its cutoff for France has to be tuned again: a blend's
probabilities are not LightGBM's.

**Run on Kaggle** with the GPU (2×T4) on. Attach the competition data, 01's output and 02 (full)'s output, then
Run all. My estimate is 03's time (about 5 h) plus 1-2 hours: each extra model is trained 6 times (3 folds,
2 country transfers, final). The comparison table is printed about halfway through the run.
* `CROSS_COUNTRY_CHECK=false` saves about a third of the extra time.
* `PREDICT_TEST=false` stops after the comparison and the transfer (no submission files).

**Settings** are 03's (first code cell, or environment variables), plus:
* `MODELS` (default `lgbm,xgb,cat,nn`)
* `ENSEMBLE`: `auto`, `mean`, `stack`, or one model
* `NN_EPOCHS`
* `CROSS_COUNTRY_CHECK` (`false` skips the transfer)
* `COUNTRY_THRESHOLDS` (JSON, e.g. `{"France": 0.9}`)

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "experiments":     # run from the repo root, like the pipeline notebooks (no-op on Kaggle)
    os.chdir(Path.cwd().parent)

import collections
import contextlib
import csv
import gc
import glob
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import warnings
from pathlib import Path

import itertools

import numpy as np
import scipy.sparse as sp
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

for pkg in ("lightgbm", "xgboost", "catboost", "rapidfuzz", "anyascii"):   # MIT, Apache 2.0, Apache 2.0, MIT, ISC
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import catboost as cb
import lightgbm as lgb
import torch
import xgboost as xgb
from rapidfuzz import fuzz
from anyascii import anyascii
from rapidfuzz import process as rf_process
from rapidfuzz.distance import JaroWinkler, Levenshtein
from sklearn.linear_model import LogisticRegression

CANDIDATES_PER_S1 = int(os.environ.get("CANDIDATES_PER_S1", 30))          # top e5 candidates judged per S1 (30 beat 20 on the test bed)
USE_EXTRA_CANDIDATES = os.environ.get("USE_EXTRA_CANDIDATES", "true") == "true"  # also judge 02's extra candidates (if present)
WORD_MAP = os.environ.get("WORD_MAP", "true") == "true"                      # map transliterated Indian-script words to English
PRED_PARTS = int(os.environ.get("PRED_PARTS", 8))                          # test: S1 groups per country, one at a time (memory)
TRAIN_S1_PER_COUNTRY = int(os.environ.get("TRAIN_S1_PER_COUNTRY", 200_000))  # train S1 used for learning, per country
N_FOLDS = 3
SEED = 42
LGBM_DEVICE = os.environ.get("LGBM_DEVICE", "auto")                        # auto | cpu | gpu | cuda
CROSS_COUNTRY_CHECK = os.environ.get("CROSS_COUNTRY_CHECK", "true") == "true"      # train on one country, score the other
PAIRS_PER_CHUNK = 2_000_000                                                 # pairs featurized at a time (memory)
FEATURE_SET = os.environ.get("FEATURE_SET", "all")                          # all | submission2 (61, without typo / candidate-vs-candidate) | base (44)
PREDICT_TEST = os.environ.get("PREDICT_TEST", "true") == "true"             # false = experiment: scores only, no test prediction / files
MODELS = [m.strip() for m in os.environ.get("MODELS", "lgbm,xgb,cat,nn").split(",") if m.strip()]  # any of lgbm, xgb, cat, nn
ENSEMBLE = os.environ.get("ENSEMBLE", "auto")                              # auto (best cross-validation score) | mean | stack | one of MODELS
NN_EPOCHS = int(os.environ.get("NN_EPOCHS", 6))                            # neural network: passes over the training pairs
COUNTRY_THRESHOLDS = json.loads(os.environ.get("COUNTRY_THRESHOLDS", "{}"))   # optional stricter cutoff per test country, e.g. {"France": 0.9}; others use the tuned t
T0 = time.time()


def search(patterns, roots):
    """First existing file matching any glob pattern under any root (shortest path wins); skips __MACOSX."""
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for pat in patterns for h in glob.glob(os.path.join(root, pat), recursive=True) if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


KAGGLE = os.path.isdir("/kaggle/working")
LOCAL_DATA = str(Path.cwd().parent / "Data" / "student_resource")
if os.environ.get("BUCKET_DIR"):
    BUCKET_DIR = Path(os.environ["BUCKET_DIR"])
else:
    hit = search(["**/buckets/train_*.parquet", "**/buckets/test_*.parquet"],
                 ["embeddings_full", "/kaggle/working/embeddings_full", "/kaggle/input"])
    assert hit is not None, "no buckets found: run 02_full_e5_buckets.ipynb (or attach its output) or set BUCKET_DIR"
    BUCKET_DIR = hit.parent
MANIFEST = json.loads((BUCKET_DIR.parent / "manifest.json").read_text(encoding="utf-8")) \
    if (BUCKET_DIR.parent / "manifest.json").exists() else {}
proc = search(["train_s1_*.parquet", "**/train_s1_*.parquet"],
              [os.environ.get("PROCESSED_DIR", ""), MANIFEST.get("settings", {}).get("processed_dir", ""), "processed",
               "/kaggle/working/processed", "/kaggle/input"])
assert proc is not None, "processed/ not found: attach 01's output or set PROCESSED_DIR"
PROCESSED_DIR = proc.parent
DATA_ROOTS = [os.environ.get("DATA_ROOT", ""), "/kaggle/input", LOCAL_DATA]
GT_FILE = search(["**/train_ground_truth.tsv"], DATA_ROOTS)
TEST_S1_FILE = search(["**/test_source1.tsv"], DATA_ROOTS)
assert GT_FILE is not None and TEST_S1_FILE is not None, "competition data not found: attach it or set DATA_ROOT"
TEST_DIR = TEST_S1_FILE.parent
VALIDATOR = Path(os.environ["VALIDATOR"]) if os.environ.get("VALIDATOR") else \
    search(["**/validate_submission.py"], ["/kaggle/input", LOCAL_DATA])
OUT_DIR = Path(os.environ.get("SUBMISSION_DIR") or ("/kaggle/working/submission" if KAGGLE else "submission"))
for sub in ("output", "model", "test_probabilities"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
print(f"buckets: {BUCKET_DIR}\nprocessed: {PROCESSED_DIR}\nground truth: {GT_FILE}\ntest data: {TEST_DIR}\n"
      f"validator: {VALIDATOR}\noutput: {OUT_DIR}")
BUCKETS = sorted(p.stem for p in BUCKET_DIR.glob("*_*.parquet"))
TRAIN_COUNTRIES = [b.split("_", 1)[1] for b in BUCKETS if b.startswith("train_")]
TEST_COUNTRIES = [b.split("_", 1)[1] for b in BUCKETS if b.startswith("test_")]
print(f"train buckets: {TRAIN_COUNTRIES} | test buckets: {TEST_COUNTRIES}")
if not PREDICT_TEST:
    TEST_COUNTRIES = []
    print("PREDICT_TEST=false: experiment run (cross-validation and scores only; no test prediction, no submission files)")
assert TRAIN_COUNTRIES and (TEST_COUNTRIES or not PREDICT_TEST),     "both train and test buckets are needed (run 02 full with SPLITS=train,test), or set PREDICT_TEST=false"


def lgbm_device():
    """GPU if this LightGBM build supports it (tried on a tiny dataset), else CPU."""
    if LGBM_DEVICE != "auto":
        return LGBM_DEVICE
    if shutil.which("nvidia-smi") is None:
        return "cpu"
    x = np.random.default_rng(0).random((2000, 4))
    for device in ("cuda", "gpu"):
        try:
            lgb.train({"objective": "binary", "device_type": device, "verbose": -1},
                      lgb.Dataset(x, (x[:, 0] > 0.5).astype(int)), num_boost_round=2)
            return device
        except Exception:
            pass
    print("this LightGBM build has no GPU support, training on the CPU (the 'Fatal' lines above come from that test)")
    return "cpu"


DEVICE = lgbm_device()
PARAMS = {"objective": "binary", "learning_rate": 0.1, "num_leaves": 63, "min_data_in_leaf": 100,
          "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
          "max_bin": 255 if DEVICE == "cpu" else 63, "device_type": DEVICE, "seed": SEED, "verbose": -1}
print(f"LightGBM {lgb.__version__} on {DEVICE}; {CANDIDATES_PER_S1} candidates per S1; "
      f"{TRAIN_S1_PER_COUNTRY:,} train S1 per country; {N_FOLDS} folds")

## The four models
| Model | Settings | Hardware |
|---|---|---|
| LightGBM (`lgbm`) | 03's: 63 leaves, learning rate 0.1 | as in 03 (the GPU if this LightGBM build supports it) |
| XGBoost (`xgb`) | depth-8 trees, learning rate 0.1, 80% of the rows and features per tree | first GPU |
| CatBoost (`cat`) | depth-8 symmetric trees, learning rate 0.1 | first GPU |
| Neural network (`nn`) | 3 layers (512, 256, 128) with batch norm and dropout, one-cycle learning rate, `NN_EPOCHS` passes | second GPU (the first if there is only one) |

**Rounds.** The boosting models stop 50 rounds after the held-out fold's log-loss stops improving. The final
model then trains for 1.1 × the folds' mean best round count, as 03 does. The network trains for `NN_EPOCHS` passes
in cross-validation and in the final model alike.

**Network inputs.** Each feature is mapped to its rank among the training pairs (-1 to 1, 0 when empty), plus a
0/1 "is empty" column for each feature that can be empty. Trees need no such mapping, but a network does: the
features range from 0/1 flags to counts and 0-100 ratios.

Without a GPU, every model trains on the CPU.

In [ ]:
N_GPUS = torch.cuda.device_count()
unknown = set(MODELS) - {"lgbm", "xgb", "cat", "nn"}
assert MODELS and not unknown, f"unknown models {unknown}: MODELS takes lgbm, xgb, cat, nn"
XGB_PARAMS = {"objective": "binary:logistic", "eval_metric": "logloss", "eta": 0.1, "max_depth": 8,
              "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 0.8, "lambda": 1.0, "max_bin": 256,
              "seed": SEED}
if int(xgb.__version__.split(".")[0]) >= 2:
    XGB_PARAMS.update(tree_method="hist", device="cuda:0" if N_GPUS else "cpu")
else:
    XGB_PARAMS.update(tree_method="gpu_hist" if N_GPUS else "hist")
CAT_PARAMS = {"loss_function": "Logloss", "learning_rate": 0.1, "depth": 8, "l2_leaf_reg": 3.0, "random_seed": SEED,
              "verbose": 0, "allow_writing_files": False}
# gpu_ram_part: CatBoost takes 95% of the GPU by default, leaving nothing for XGBoost's memory cache
CAT_PARAMS.update({"task_type": "GPU", "devices": "0", "gpu_ram_part": 0.6} if N_GPUS else {"task_type": "CPU", "thread_count": -1})
NN_DEVICE = torch.device(f"cuda:{min(1, N_GPUS - 1)}" if N_GPUS else "cpu")
NN_AMP = NN_DEVICE.type == "cuda"                  # half precision on the GPU
NN_BATCH, NN_LR, NN_HIDDEN = 8192, 2e-3, (512, 256, 128)
MAX_ROUNDS, EARLY_STOP = 3000, 50
XN = YN = None                                     # network inputs and labels of the training pairs, on NN_DEVICE
warnings.filterwarnings("ignore", message="Detected call of `lr_scheduler.step()`")   # a skipped half-precision step
print(f"{N_GPUS} GPU(s) | models: {', '.join(MODELS)} | XGBoost {xgb.__version__} on "
      f"{XGB_PARAMS.get('device', XGB_PARAMS['tree_method'])} | CatBoost {cb.__version__} on {CAT_PARAMS['task_type']} | "
      f"network: PyTorch {torch.__version__} on {NN_DEVICE}")


def fit(x_train, y_train, x_valid=None, y_valid=None, rounds=2000):
    """Train LightGBM; with a validation set, stop 50 rounds after its log-loss stops improving."""
    train_set = lgb.Dataset(x_train, y_train, feature_name=FEATURES)
    if x_valid is None:
        return lgb.train(PARAMS, train_set, rounds)
    valid_set = lgb.Dataset(x_valid, y_valid, reference=train_set)
    return lgb.train(PARAMS, train_set, rounds, valid_sets=[valid_set],
                     callbacks=[lgb.early_stopping(50, verbose=False)])


def xgb_matrix(x, label, ref=None):
    """XGBoost training data (quantized when this version can, which saves memory)."""
    if hasattr(xgb, "QuantileDMatrix"):
        return xgb.QuantileDMatrix(x, label, max_bin=XGB_PARAMS["max_bin"], ref=ref)
    return xgb.DMatrix(x, label)


class MLP(torch.nn.Module):
    """Feed-forward network: NN_HIDDEN layers with batch norm, SiLU and dropout; outputs the log-odds."""

    def __init__(self, d):
        super().__init__()
        layers = []
        for h in NN_HIDDEN:
            layers += [torch.nn.Linear(d, h), torch.nn.BatchNorm1d(h), torch.nn.SiLU(), torch.nn.Dropout(0.1)]
            d = h
        self.net = torch.nn.Sequential(*layers, torch.nn.Linear(d, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)


def amp():
    """Half-precision context on the GPU, nothing on the CPU."""
    return torch.autocast("cuda", dtype=torch.float16) if NN_AMP else contextlib.nullcontext()


def grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=NN_AMP)
    except (AttributeError, TypeError):            # PyTorch < 2.3
        return torch.cuda.amp.GradScaler(enabled=NN_AMP)


def nn_fit_transform(x, n_sample=500_000):
    """Per feature, the quantiles of a sample of the training pairs (for the rank mapping); and the features that
    can be empty."""
    rows = np.sort(np.random.default_rng(SEED).choice(len(x), min(n_sample, len(x)), replace=False))
    sample, levels = x[rows], np.linspace(0, 1, 201)
    quantiles, can_be_empty = [], []
    for j in range(x.shape[1]):
        col = sample[:, j][~np.isnan(sample[:, j])]
        if len(col):
            q, first = np.unique(np.quantile(col, levels), return_index=True)
            quantiles.append((q, levels[first]))
        else:
            quantiles.append((np.zeros(1), np.full(1, 0.5)))
        if np.isnan(x[:, j]).any():
            can_be_empty.append(j)
    return quantiles, np.array(can_be_empty, int)


def nn_matrix(x):
    """Network inputs for the feature rows x, on NN_DEVICE: every feature mapped to its rank among the training
    pairs (-1 to 1, 0 when empty), then a 0/1 column for every feature that can be empty."""
    out = torch.empty((len(x), len(NN_Q) + len(NN_EMPTY)), dtype=torch.float16 if NN_AMP else torch.float32,
                      device=NN_DEVICE)
    for j, (q, levels) in enumerate(NN_Q):
        col = x[:, j]
        out[:, j] = torch.from_numpy(np.where(np.isnan(col), 0.0, 2 * np.interp(col, q, levels) - 1)
                                     .astype(np.float32)).to(NN_DEVICE)
    for k, j in enumerate(NN_EMPTY):
        out[:, len(NN_Q) + k] = torch.from_numpy(np.isnan(x[:, j]).astype(np.float32)).to(NN_DEVICE)
    return out


@torch.no_grad()
def nn_predict(net, xn, rows=None, batch=262_144):
    """Probabilities of the network for the rows (indices; None = all) of the input matrix xn."""
    net.eval()
    n = len(xn) if rows is None else len(rows)
    rows = None if rows is None else torch.as_tensor(rows, device=xn.device)
    out = []
    for s in range(0, n, batch):
        with amp():
            out.append(torch.sigmoid(net(xn[s:s + batch] if rows is None else xn[rows[s:s + batch]]).float()))
    return torch.cat(out).cpu().numpy() if out else np.zeros(0, np.float32)


def nn_fit(tr, va=None, epochs=None):
    """Train the network on the training rows tr (indices into XN); with held-out rows va, print their log-loss
    after every pass."""
    epochs = epochs or NN_EPOCHS
    torch.manual_seed(SEED)
    net = MLP(XN.shape[1]).to(NN_DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=NN_LR, weight_decay=1e-5)
    tr = torch.as_tensor(tr, device=NN_DEVICE)
    batch = int(np.clip(len(tr) // 200, 256, NN_BATCH))   # at least ~200 steps per pass on small data
    with torch.no_grad():                                  # start from the share of true pairs, not 50%
        net.net[-1].bias.fill_(float(logit(YN[tr].mean().item())))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=NN_LR, pct_start=0.1,
                                                total_steps=epochs * math.ceil(len(tr) / batch))
    scaler = grad_scaler()
    for epoch in range(epochs):
        net.train()
        order = tr[torch.randperm(len(tr), device=NN_DEVICE)]
        for s in range(0, len(order), batch):
            rows = order[s:s + batch]
            if len(rows) < 2:                      # batch norm needs two rows
                continue
            with amp():
                loss = torch.nn.functional.binary_cross_entropy_with_logits(net(XN[rows]), YN[rows])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
        if va is not None:
            print(f"    network pass {epoch + 1}/{epochs}: held-out log-loss {logloss(y[va], nn_predict(net, XN, va)):.4f}",
                  flush=True)
    return net


def fit_model(name, tr=None, va=None, rounds=None):
    """Train one model on the training pairs selected by the mask tr (None = all), stopping early on the held-out
    rows va, or for `rounds` rounds (passes for the network). Returns (model, rounds used)."""
    if name == "nn":
        rows = np.arange(len(y)) if tr is None else np.flatnonzero(tr)
        return nn_fit(rows, None if va is None else np.flatnonzero(va), rounds), rounds or NN_EPOCHS
    x_tr, y_tr = (X, y) if tr is None else (X[tr], y[tr])
    if name == "lgbm":
        if va is None:
            return fit(x_tr, y_tr, rounds=rounds), rounds
        model = fit(x_tr, y_tr, X[va], y[va])
        return model, model.best_iteration
    if name == "xgb":
        train_set = xgb_matrix(x_tr, y_tr)
        del x_tr
        if va is None:
            return xgb.train(XGB_PARAMS, train_set, rounds), rounds
        model = xgb.train(XGB_PARAMS, train_set, MAX_ROUNDS, evals=[(xgb_matrix(X[va], y[va], ref=train_set), "valid")],
                          early_stopping_rounds=EARLY_STOP, verbose_eval=False)
        return model, model.best_iteration + 1
    train_set = cb.Pool(x_tr, y_tr)
    del x_tr
    if va is None:
        model = cb.CatBoostClassifier(iterations=rounds, **CAT_PARAMS)
        model.fit(train_set)
        return model, rounds
    model = cb.CatBoostClassifier(iterations=MAX_ROUNDS, **CAT_PARAMS)
    model.fit(train_set, eval_set=cb.Pool(X[va], y[va]), early_stopping_rounds=EARLY_STOP)
    return model, model.get_best_iteration() + 1


def predict_model(name, model, rounds, x=None, rows=None):
    """One model's probabilities for the feature matrix x (test chunks), or for the training pairs selected by the
    mask rows."""
    if name == "nn":
        if x is None:
            return nn_predict(model, XN, np.flatnonzero(rows))
        xn = nn_matrix(x)
        p = nn_predict(model, xn)
        del xn
        return p
    x = X[rows] if x is None else x
    if name == "lgbm":
        return model.predict(x, num_iteration=rounds).astype(np.float32)
    if name == "xgb":
        return model.predict(xgb.DMatrix(x), iteration_range=(0, rounds)).astype(np.float32)
    return model.predict_proba(x)[:, 1].astype(np.float32)


def logloss(y_true, p):
    p = np.clip(np.asarray(p, np.float64), 1e-7, 1 - 1e-7)
    return float(-np.mean(np.where(y_true, np.log(p), np.log(1 - p))))


def logit(p):
    p = np.clip(np.asarray(p, np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def free_gpu():
    gc.collect()
    if N_GPUS:
        torch.cuda.empty_cache()

## Challenge metric: macro F0.5 per S1
For every S1, F0.5 over its predicted vs true S2/S3 IDs, averaged over all S1. An S1 with no true
match scores 1.0 for an empty prediction and 0.0 otherwise.

In [ ]:
def f05(predicted, true):
    """Challenge F0.5 for one S1: predicted and true are collections of S2/S3 entity IDs."""
    predicted, true = set(predicted), set(true)
    if not true:
        return 1.0 if not predicted else 0.0
    tp = len(predicted & true)
    if tp == 0:
        return 0.0
    precision, recall = tp / len(predicted), tp / len(true)
    return 1.25 * precision * recall / (0.25 * precision + recall)


def macro_f05(pred_by_s1, true_by_s1, s1_ids):
    """Mean challenge F0.5 over s1_ids (S1 missing from pred_by_s1 = empty prediction)."""
    return float(np.mean([f05(pred_by_s1.get(s, ()), true_by_s1.get(s, ())) for s in s1_ids]))


assert round(f05(["S2-00047", "S2-00193", "S3-00812"], ["S2-00047", "S3-00812"]), 3) == 0.714
print("challenge metric OK (problem-statement example = 0.714)")

## Records and features
The 44 features of notebook 03, 13 that separate lookalikes, 8 typo-tolerant versions of the word
rarity features, 7 that say which search found the pair and 13 that compare a candidate with the
other candidates of the same S1 (85 in total), computed from 01's processed fields and the buckets:

| Group | Features |
|---|---|
| Embedding | rank, retrieval score, cosine; gap to the S1's best candidate and to the next one; the S1's best and top-5 cosine |
| Other S1s | how many S1 buckets hold this candidate, whether this S1 is its best S1, and the score margin to its other best S1 |
| Name | edit-distance ratios, token-set/sort ratios, Jaro-Winkler, transliterated-name ratios, compact name/domain ratio, phonetic code, legal form, lengths |
| Address | ratios on the cleaned and raw address, shared street tokens, shared numbers, postcode, house number, city, state |
| Candidate flags | S2 or S3, native script, web name, empty address, honorific, landmark |
| **Street** (new) | the street name (the address part with the house number, minus numbers, street words like rue/avenue/road and stop words) compared by token-set and plain ratio; the house number (leading zeros removed) |
| **Word rarity** (new) | name and address word overlap weighted by how rare each word is among the S1 of that country (IDF), the share of the S1's weight found in the candidate, and the rarest unmatched word on each side |
| **Typo-tolerant word rarity** | the 8 word-rarity features again, with words that differ by a typo counted as shared (at most 1 letter for 4-5 letter words, 2 for longer ones; numbers must match exactly). Kept next to the exact ones: on the test bed, exact + tolerant gave 0.9675 vs 0.9667, and tolerant alone 0.9659 |
| **Candidate vs candidate** | for address, address word rarity, street name, name and name word rarity: how far this candidate is behind the best candidate of the same S1, and its rank among them; how many candidates have a near-identical name / address (crowding); how many have an equally close name and a clearly better address. On the test bed: 0.9667 → 0.9668, and the cross-country check (the stand-in for France) 0.8761 → 0.8789 (US → India) and 0.9119 → 0.9123 (India → US) |
| **Name frequency** (new) | how many S1 per 100,000 in that country have the S1's name, and the candidate's name |
| **Found by** (with 02's extra sources) | rank in the e5 search, rank of this S1 among the candidate's best S1 (reverse search), same house number + street word, rank among the S1's most similar names by 3-grams; same number + address word (`num_key`), and for candidates without an address the rank in the name search over them, from the S1's side and from the candidate's (`empty_rank`, `empty_rev_rank`); empty when that search did not find the pair (and when 02 ran without that source) |

Why the new ones: France's S1 sit in about 15 cities and their names are mostly a city plus a
generic word ("Bordeaux Club" is one of the most common). Lookalikes there share the city, house
number, a generic word and the legal form, but not the street or the rare name word. Rarity is
measured per country (only from unlabeled S1 records), so "club" weighs little in France.

Country is not a feature, so the model applies to France. Where a field is empty on either side,
the feature is left empty (NaN) and LightGBM treats it as unknown.

**Indian scripts:** 01's cleaning strips combining marks, which deletes the vowel signs of Indian
scripts, so its `name_norm`, `name_lat` and `addr_norm` are garbled for such records. For every
name or address with a native script, all comparisons here use the original text transliterated
with `anyascii` instead ("సన్ ఎనర్జీ లిమిటెడ్" -> "sn enrji limited"), with the same two fixes as 02
(Malayalam "tt", nasal "n") and, for names, the same word map ("sn enrji limited" -> "sun energy limited";
`translit_wordmap.json` from 02's output, or learned here from the train pairs when it is missing), as 02 (full) does for the
embeddings. Latin-script records are unchanged.

In [ ]:
try:
    from rapidfuzz.process import cpdist   # rapidfuzz >= 3.6: many pairs at once on all CPU cores
except ImportError:
    cpdist = None

STR_COLS = ["business_name", "business_address", "name_norm", "addr_norm", "core_name", "name_lat", "legal_form",
            "domain_stem", "phonetic_code", "street_tokens", "house_no", "city", "state_code"]
FLAG_COLS = ["has_honorific", "empty_addr", "landmark_flag"]
TAGS = ["tag_native_script", "tag_web_name", "tag_empty_address", "tag_honorific", "tag_landmark"]
# a name has a native script when it has a character outside Latin, Latin-1/Extended and general punctuation
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")
# Two fixes to anyascii for Indian scripts (measured on true pairs, Latin S1 name vs Indian-script candidate name,
# mean token-set similarity: Malayalam 66.1 -> 72.5, Telugu 75.5 -> 76.7):
# * Malayalam റ്റ is read "tt"; anyascii gives "rr" ("ലിമിറ്റഡ്" -> "limirrd" instead of "limittd").
# * An anusvara (the nasal dot) before a consonant other than p/ph/b/bh/m is read "n"; anyascii always gives "m"
#   ("आनंद" -> "anmd" instead of "annd", "सेंटर" -> "semtr" instead of "sentr").
_INDIC_BLOCKS = range(0x0900, 0x0D80, 0x80)                     # Devanagari, Bengali, ..., Malayalam (same layout)
_ANUSVARA = "".join(chr(b + 0x02) for b in _INDIC_BLOCKS)
_LABIALS = "".join(chr(b + o) for b in _INDIC_BLOCKS for o in range(0x2A, 0x2F))      # pa, pha, ba, bha, ma
RE_ANUSVARA_N = re.compile(f"[{_ANUSVARA}](?=[\u0900-\u0D7F])(?![{_LABIALS}])")


def translit(s):
    """anyascii with the two fixes above."""
    return anyascii(RE_ANUSVARA_N.sub("n", s.replace("\u0D31\u0D4D\u0D31", "\u0D1F\u0D4D\u0D1F")))
RE_HANDLE = re.compile(r"^\s*[@#]\S+\s*$")
RE_NUM = re.compile(r"\d+")
RE_POSTCODE = re.compile(r"(?<!\d)\d{5,6}(?!\d)")
RE_NOT_ALNUM = re.compile(r"[^0-9a-z]+")
FEATURES = ["rank", "score", "cosine", "s1_best_cosine", "gap_to_best_score", "gap_to_next_score", "s1_top5_cosine",
            "cand_n_s1", "cand_rank", "cand_margin", "name_ratio", "name_token_set", "name_token_sort", "name_partial",
            "core_ratio", "core_jaro_winkler", "latin_ratio", "latin_token_set", "raw_name_token_set", "compact_ratio",
            "compact_partial", "phonetic_same", "legal_form_same", "s1_name_len", "cand_name_len", "addr_ratio",
            "addr_token_set", "raw_addr_token_set", "raw_addr_partial", "street_jaccard", "street_cover",
            "numbers_jaccard", "numbers_cover", "postcode_same", "house_no_same", "city_same", "state_same",
            "cand_addr_len", "cand_is_s3", "cand_native_script", "cand_web_name", "cand_empty_address",
            "cand_honorific", "cand_landmark"]
BASE_FEATURES = list(FEATURES)
FEATURES = FEATURES + ["name_idf_jaccard", "name_idf_cover", "name_idf_miss_s1", "name_idf_miss_cand", "addr_idf_jaccard",
                       "addr_idf_cover", "addr_idf_miss_s1", "addr_idf_miss_cand", "street_name_token_set",
                       "street_name_ratio", "house_number_same", "s1_name_freq", "cand_name_freq"]
FEATURES = FEATURES + [f"{k}_idf_{c}_fuzzy" for k in ("name", "addr") for c in ("jaccard", "cover", "miss_s1", "miss_cand")]
REL_SIMS = ["addr_token_set", "addr_idf_cover", "street_name_token_set", "name_token_set", "name_idf_cover"]
REL_FEATURES = [f"{c}_{k}" for c in REL_SIMS for k in ("gap_best", "rank")] + ["n_name_twins", "n_addr_twins",
                                                                               "better_addr_same_name"]
FEATURES = FEATURES + REL_FEATURES
NEW_SOURCE_FEATURES = ["num_key", "empty_rank", "empty_rev_rank"]               # 02's number and empty sources
SOURCE_FEATURES = ["e5_rank", "rev_rank", "addr_key", "name_rank"] + NEW_SOURCE_FEATURES   # written by 02 with extra sources
FEATURES = FEATURES + SOURCE_FEATURES
if FEATURE_SET == "base":
    FEATURES = BASE_FEATURES
elif FEATURE_SET == "submission2":                                  # the 61 features of submission 2 (for comparisons)
    FEATURES = [f for f in FEATURES if f not in REL_FEATURES + NEW_SOURCE_FEATURES and not f.endswith("_fuzzy")]
assert FEATURE_SET in {"all", "submission2", "base"}, FEATURE_SET
_ACCENTS = {}
for _cp in range(0xC0, 0x250):
    _base = unicodedata.normalize("NFKD", chr(_cp))[0]
    if _base.isascii() and _base != chr(_cp):
        _ACCENTS[_cp] = _base
FOLD = str.maketrans(_ACCENTS)
RE_WORD = re.compile(r"\w+")
STREET_TYPES = frozenset("rue avenue av ave boulevard bd blvd boul chemin che place pl route rte allee impasse imp quai "
                         "cours crs square faubourg fbg road rd street st lane ln drive dr way court ct circle cir "
                         "highway hwy parkway pkwy terrace marg".split())
STREET_STOP = STREET_TYPES | frozenset("r ch all sq bis ter de du des la le les l d of the and et en sur suite ste unit "
                                       "apt apartment building bldg floor fl po box door no nr n number h hno flat shop "
                                       "plot near opp".split())


# Word map for Indian scripts: most Indian-script business names are English words written in that script, and
# their transliterations are misspelled English ("praivet limited", "solyusns", "injiniyring"). The map sends such
# words back to English ("private limited", "solutions", "engineering"). It is learned from the train true pairs
# whose candidate name is in an Indian script: each transliterated word is lined up with the most similar word of
# its true S1's name (rapidfuzz ratio >= 55), and kept when the same English word wins in at least
# WORDMAP_MIN_PAIRS pairs and WORDMAP_MIN_SHARE of the word's occurrences. Test bed (map learned without its S1):
# name similarity of Indian-script true pairs 72-77 -> 88-92; matcher 0.9667 -> 0.9675, country transfer +0.003-0.005.
WORDMAP_MIN_PAIRS, WORDMAP_MIN_SHARE, WORDMAP_MIN_RATIO = 20, 0.6, 55
_RE_WORD2 = re.compile(r"[a-z]{2,}")


def learn_word_map(processed_dir, gt_file):
    """{transliterated word: English word} from the train true pairs of every country (only Indian-script names count)."""
    gt = pd.read_csv(gt_file, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
    gp = gt[gt["matched_entity_ids"] != ""]
    gp = gp.assign(cand=gp["matched_entity_ids"].str.split(",")).explode("cand")
    owner = dict(zip(gp["cand"], gp["source1_entity_id"]))
    del gt, gp
    s1_name = {}
    for f in sorted(Path(processed_dir).glob("train_s1_*.parquet")):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        s1_name.update(zip(d["entity_id"], d["business_name"].fillna("").astype(str)))
    pair_counts, tok_counts = collections.Counter(), collections.Counter()
    for f in sorted(list(Path(processed_dir).glob("train_s2_*.parquet")) + list(Path(processed_dir).glob("train_s3_*.parquet"))):
        d = pd.read_parquet(f, columns=["entity_id", "business_name"])
        for e, name in zip(d["entity_id"], d["business_name"].fillna("").astype(str)):
            s = owner.get(e)
            if s is None or s not in s1_name or not RE_NATIVE.search(name):
                continue
            a = set(_RE_WORD2.findall(s1_name[s].lower()))
            for t in set(_RE_WORD2.findall(translit(name).lower())):
                tok_counts[t] += 1
                if t in a or not a:
                    continue
                best = rf_process.extractOne(t, a, scorer=fuzz.ratio, score_cutoff=WORDMAP_MIN_RATIO)
                if best:
                    pair_counts[(t, best[0])] += 1
    best_of = {}
    for (t, e), n in pair_counts.items():
        if n > best_of.get(t, (0, ""))[0] or (n == best_of[t][0] and e < best_of[t][1]):
            best_of[t] = (n, e)
    return {t: e for t, (n, e) in sorted(best_of.items())
            if n >= WORDMAP_MIN_PAIRS and n / tok_counts[t] >= WORDMAP_MIN_SHARE}


def translit_name(s):
    """Transliterated Indian-script name with the word map applied ('प्राइवेट लिमिटेड' -> 'private limited')."""
    return " ".join(WORDMAP.get(w, w) for w in translit(s).lower().split())



WORDMAP_FILE = next((p for p in (BUCKET_DIR.parent / "translit_wordmap.json",              # written by 02 (full):
                                 BUCKET_DIR.parent.parent / "translit_wordmap.json") if p.exists()),   # next to or above
                    BUCKET_DIR.parent / "translit_wordmap.json")                              # the model folder
if not WORD_MAP:
    WORDMAP = {}
elif WORDMAP_FILE.exists():
    WORDMAP = json.loads(WORDMAP_FILE.read_text(encoding="utf-8"))
    print(f"word map: {len(WORDMAP)} words from {WORDMAP_FILE}")
else:                                                              # buckets from an older 02: learn it the same way
    _t = time.time()
    WORDMAP = learn_word_map(PROCESSED_DIR, GT_FILE)
    (OUT_DIR / "model" / "translit_wordmap.json").write_text(json.dumps(WORDMAP, ensure_ascii=False, indent=0, sort_keys=True),
                                                            encoding="utf-8")
    print(f"word map: {len(WORDMAP)} words learned from the train pairs in {time.time() - _t:.0f}s")


def fold_text(s):
    """Lower case with Latin accents removed (other scripts unchanged)."""
    return s.lower() if s.isascii() else s.lower().translate(FOLD)


def words(s):
    """Folded word tokens; numbers without leading zeros ('0058' -> '58')."""
    return [(w.lstrip("0") or "0") if w.isdigit() else w for w in RE_WORD.findall(fold_text(s))]


def street_of(address):
    """(street name, house number): the first comma part with a number or a street word, minus numbers, street
    words and stop words. '15 bis Rue de Barbieux, Roubaix' -> ('barbieux', '15')."""
    house = ""
    for part in address.split(","):
        w = words(part)
        nums = [x for x in w if x.isdigit()]
        if nums and not house:
            house = nums[0]
        if nums or any(x in STREET_TYPES for x in w):
            name = " ".join(x for x in w if not x.isdigit() and x not in STREET_STOP and len(x) > 1)
            if name:
                return name, house
    return "", house


def country_stats(split, country):
    """Word rarity (IDF) in the names and addresses of all S1 of one split and country, and how many S1 share each name."""
    s1 = pd.read_parquet(PROCESSED_DIR / f"{split}_s1_{country}.parquet", columns=["core_name", "business_address"])
    names = [frozenset(words(x)) for x in s1["core_name"].fillna("").astype(str)]
    addrs = [frozenset(words(x)) for x in s1["business_address"].fillna("").astype(str)]
    n = len(s1)

    def idf(sets):
        df = collections.Counter(t for s in sets for t in s)
        return {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in df.items()}

    return {"name_idf": idf(names), "addr_idf": idf(addrs), "unseen_idf": math.log(n + 1) + 1.0, "n": n,
            "name_freq": collections.Counter(" ".join(sorted(s)) for s in names)}


def _indicator_rows(a, b):
    """Sparse 0/1 rows (one per pair) for two aligned arrays of sets, over their shared vocabulary."""
    codes_a, sets_a = pd.factorize(np.asarray(a, dtype=object))
    codes_b, sets_b = pd.factorize(np.asarray(b, dtype=object))
    len_a = np.fromiter(map(len, sets_a), np.int64, len(sets_a))
    len_b = np.fromiter(map(len, sets_b), np.int64, len(sets_b))
    n_a = int(len_a.sum())
    flat = np.fromiter(itertools.chain.from_iterable(itertools.chain(sets_a, sets_b)), object, n_a + int(len_b.sum()))
    word_ids, vocab = pd.factorize(flat)
    n_words = max(len(vocab), 1)
    ma = sp.csr_matrix((np.ones(n_a), word_ids[:n_a], np.r_[0, np.cumsum(len_a)]), shape=(len(sets_a), n_words))[codes_a]
    mb = sp.csr_matrix((np.ones(len(word_ids) - n_a), word_ids[n_a:], np.r_[0, np.cumsum(len_b)]),
                       shape=(len(sets_b), n_words))[codes_b]
    return ma, mb, vocab


def _near(w, v):
    """Two alphabetic words of 4+ letters that differ by a typo: at most 1 edit (4-5 letters) or 2 (6+), length
    difference at most 2."""
    if min(len(w), len(v)) < 4 or abs(len(w) - len(v)) > 2 or not (w.isalpha() and v.isalpha()):
        return False
    k = 1 if len(w) < 6 else 2
    return Levenshtein.distance(w, v, score_cutoff=k) <= k


def weighted_overlap(a, b, idf, unseen):
    """For aligned arrays of word sets (NaN when either side is empty), rows 0-3: IDF-weighted Jaccard, share of the
    S1's weight found in the candidate, and the rarest unmatched word on each side; rows 4-7: the same with words
    that differ by a typo counted as shared (see _near; a typo pair weighs the S1 word's weight on both sides).
    Rows 0-3 use sparse matrices (weights summed / maximized per pair in C); the typo search loops only over
    pairs where both sides have an unmatched word that could be a typo."""
    out = np.full((8, len(a)), np.nan, np.float32)
    if not len(a):
        return out
    ma, mb, vocab = _indicator_rows(a, b)
    w = np.fromiter((idf.get(t, unseen) for t in vocab), np.float64, len(vocab))
    inter = ma.multiply(mb).tocsr()
    wi, wx, wy = inter @ w, ma @ w, mb @ w
    ok = (np.diff(ma.indptr) > 0) & (np.diff(mb.indptr) > 0)
    only_a, only_b = (ma - inter).tocsr(), (mb - inter).tocsr()
    only_a.eliminate_zeros()
    only_b.eliminate_zeros()
    with np.errstate(divide="ignore", invalid="ignore"):
        out[0, ok] = (wi / (wx + wy - wi))[ok]
        out[1, ok] = (wi / wx)[ok]
    out[2, ok] = only_a.multiply(w).tocsr().max(axis=1).toarray().ravel()[ok]
    out[3, ok] = only_b.multiply(w).tocsr().max(axis=1).toarray().ravel()[ok]
    out[4:, ok] = out[:4, ok]                                        # typo version: same unless a typo pair is found
    # typo search: only words that can be typos (alphabetic, 4+ letters) and only pairs with such words on both sides
    typo_able = np.fromiter((len(t) >= 4 and t.isalpha() for t in vocab), bool, len(vocab))
    ta = only_a.multiply(typo_able).tocsr()
    tb = only_b.multiply(typo_able).tocsr()
    ta.eliminate_zeros()
    tb.eliminate_zeros()
    rows = np.flatnonzero(ok & (np.diff(ta.indptr) > 0) & (np.diff(tb.indptr) > 0))
    words_of = np.asarray(vocab, dtype=object)
    near_cache = {}
    for i in rows:
        xs = ta.indices[ta.indptr[i]:ta.indptr[i + 1]]
        ys = list(tb.indices[tb.indptr[i]:tb.indptr[i + 1]])
        fi, fy, hit_any = wi[i], wy[i], False
        matched_x = set()
        for x in sorted(xs, key=lambda j: (-w[j], words_of[j])):    # rarest first
            for y in ys:
                key = (x, y)
                near = near_cache.get(key)
                if near is None:
                    near = near_cache[key] = _near(words_of[x], words_of[y])
                if near:
                    fi += w[x]
                    fy += w[x] - w[y]
                    ys.remove(y)
                    matched_x.add(x)
                    hit_any = True
                    break
        if not hit_any:
            continue
        rest_x = [j for j in only_a.indices[only_a.indptr[i]:only_a.indptr[i + 1]] if j not in matched_x]
        rest_y = set(ys) | set(only_b.indices[only_b.indptr[i]:only_b.indptr[i + 1]]) - set(tb.indices[tb.indptr[i]:tb.indptr[i + 1]])
        out[4, i] = fi / max(wx[i] + fy - fi, 1e-9)
        out[5, i] = min(fi / wx[i], 1.0)
        out[6, i] = max((w[j] for j in rest_x), default=0.0)
        out[7, i] = max((w[j] for j in rest_y), default=0.0)
    return out


def load_records(split, country, ids):
    """Fields of the given entity IDs from processed/<split>_s{1,2,3}_<country>.parquet, plus per-record helpers."""
    wanted = pa.array(sorted(ids), pa.large_string())
    frames = []
    for src in ("s1", "s2", "s3"):
        f = PROCESSED_DIR / f"{split}_{src}_{country}.parquet"
        if not f.exists():
            continue
        names = set(pq.read_schema(f).names)
        t = pq.read_table(f, columns=["entity_id"] + [c for c in STR_COLS + FLAG_COLS if c in names])
        t = t.filter(pc.is_in(t["entity_id"].cast(pa.large_string()), value_set=wanted))
        frames.append(t.to_pandas())
    ent = pd.concat(frames, ignore_index=True).drop_duplicates("entity_id").set_index("entity_id")
    for c in STR_COLS:
        ent[c] = ent[c].fillna("").astype(str) if c in ent else ""
    for c in FLAG_COLS:
        ent[c] = pd.to_numeric(ent[c], errors="coerce").fillna(0).astype(np.int8) if c in ent else 0
    name = ent["business_name"].tolist()
    ent["tag_native_script"] = [RE_NATIVE.search(x) is not None for x in name]
    # native-script names/addresses: every comparison uses the original text transliterated (01's is garbled)
    native_name = ent["tag_native_script"].to_numpy()
    if native_name.any():
        lat = [translit_name(x) for x in ent.loc[native_name, "business_name"]]
        for c in ("business_name", "name_norm", "core_name", "name_lat"):
            ent.loc[native_name, c] = lat
    native_addr = np.array([RE_NATIVE.search(x) is not None for x in ent["business_address"]])
    if native_addr.any():
        lat = [translit(x) for x in ent.loc[native_addr, "business_address"]]
        ent.loc[native_addr, "business_address"] = lat
        ent.loc[native_addr, "addr_norm"] = [" ".join(x.lower().replace(",", " ").split()) for x in lat]
    ent["tag_web_name"] = (ent["domain_stem"] != "").to_numpy() | np.array([RE_HANDLE.match(x) is not None for x in name])
    ent["tag_empty_address"] = (ent["empty_addr"] == 1) | (ent["addr_norm"] == "")
    ent["tag_honorific"] = ent["has_honorific"] == 1
    ent["tag_landmark"] = ent["landmark_flag"] == 1
    ent["raw_name"] = ent["business_name"].str.lower()
    ent["raw_addr"] = ent["business_address"].str.lower()
    ent["compact"] = [RE_NOT_ALNUM.sub("", (d or l or n).lower())
                      for d, l, n in zip(ent["domain_stem"], ent["name_lat"], ent["name_norm"])]
    ent["name_words"] = [frozenset(words(x)) for x in ent["core_name"]]
    ent["addr_words"] = [frozenset(words(x)) for x in ent["business_address"]]
    ent["numbers"] = [frozenset(x for x in w if x.isdigit()) for w in ent["addr_words"]]   # '0058' == '58'
    streets = [street_of(x) for x in ent["business_address"]]
    ent["street_name"] = [s for s, _ in streets]
    ent["house_number"] = [h for _, h in streets]
    ent["name_key"] = [" ".join(sorted(w)) for w in ent["name_words"]]
    ent["postcode"] = [(RE_POSTCODE.findall(a) or [""])[-1] for a in ent["raw_addr"]]
    ent["street_set"] = [frozenset(s.split()) for s in ent["street_tokens"]]
    return ent.drop(columns=["business_name", "business_address", "street_tokens", "domain_stem"] + FLAG_COLS)


def read_buckets(split, country):
    """Bucket rows of one (split, country), grouped by S1 in rank order, with candidate-side stats: the top
    CANDIDATES_PER_S1 e5 candidates, plus 02's extra candidates when present (and USE_EXTRA_CANDIDATES)."""
    path = BUCKET_DIR / f"{split}_{country}.parquet"
    if "e5_rank" in pq.read_schema(path).names:                      # 02 with extra sources: rank = position after filtering
        b = pq.read_table(path).to_pandas()
        e5 = b["e5_rank"].to_numpy()
        keep = (e5 < CANDIDATES_PER_S1) | (np.isnan(e5) & USE_EXTRA_CANDIDATES)
        b = b[keep].reset_index(drop=True)
        s1 = b["s1_entity_id"].to_numpy()
        starts = np.r_[0, np.flatnonzero(s1[1:] != s1[:-1]) + 1]
        b["rank"] = (np.arange(len(b)) - np.repeat(starts, np.diff(np.r_[starts, len(b)]))).astype(np.int16)
    else:
        b = pq.read_table(path, filters=[("rank", "<", CANDIDATES_PER_S1)]).to_pandas()
    s1 = b["s1_entity_id"].to_numpy()
    assert int((s1[1:] != s1[:-1]).sum()) + 1 == b["s1_entity_id"].nunique(), "bucket rows are not grouped by S1"
    # how the candidate scores against every S1 of the country that has it in its bucket
    codes, uniq = pd.factorize(b["candidate_entity_id"])
    score = b["score"].to_numpy(np.float32)
    order = np.lexsort((-score, codes))
    sc, ss = codes[order], score[order]
    start = np.r_[True, sc[1:] != sc[:-1]]
    r_sorted = np.arange(len(sc)) - np.maximum.accumulate(np.where(start, np.arange(len(sc)), 0))
    cand_rank = np.empty(len(b), np.float32)
    cand_rank[order] = r_sorted
    best = np.full(len(uniq), np.nan, np.float32)
    best[sc[start]] = ss[start]
    second = np.full(len(uniq), np.nan, np.float32)
    second[sc[r_sorted == 1]] = ss[r_sorted == 1]
    b["cand_n_s1"] = np.bincount(codes, minlength=len(uniq)).astype(np.float32)[codes]
    b["cand_rank"] = cand_rank
    b["cand_margin"] = np.where(cand_rank == 0, score - second[codes], score - best[codes]).astype(np.float32)
    return b


def chunk_bounds(b):
    """(start, end) row ranges of about PAIRS_PER_CHUNK pairs that never split an S1's bucket."""
    s1 = b["s1_entity_id"].to_numpy()
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    idx = np.searchsorted(starts, np.arange(PAIRS_PER_CHUNK, len(b), PAIRS_PER_CHUNK))
    cuts = np.unique(np.r_[0, starts[idx[idx < len(starts)]], len(b)])
    return list(zip(cuts[:-1], cuts[1:]))


def sim(scorer, a, b):
    """rapidfuzz similarity of aligned string arrays; NaN where either side is empty."""
    if cpdist is not None:
        out = cpdist(list(a), list(b), scorer=scorer, workers=-1).astype(np.float32)
    else:
        out = np.array([scorer(x, y) for x, y in zip(a, b)], np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def same(a, b):
    """1 when equal, 0 when different, NaN when either side is empty."""
    out = (a == b).astype(np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def set_overlap(a, b):
    """Jaccard overlap and share of a's items found in b, for aligned arrays of frozensets (NaN when both empty)."""
    jac = np.full(len(a), np.nan, np.float32)
    cover = np.full(len(a), np.nan, np.float32)
    if not len(a):
        return jac, cover
    ma, mb, _ = _indicator_rows(a, b)
    na, nb = np.diff(ma.indptr), np.diff(mb.indptr)
    inter = np.asarray(ma.multiply(mb).sum(axis=1)).ravel()
    both, one = (na > 0) & (nb > 0), (na > 0) ^ (nb > 0)
    jac[both] = (inter / np.maximum(na + nb - inter, 1))[both]
    cover[both] = (inter / np.maximum(na, 1))[both]
    jac[one] = cover[one] = 0.0
    return jac, cover


def make_features(b, ent, stats):
    """float32 feature matrix (columns FEATURES) for the bucket rows b (whole S1 buckets, in rank order)."""
    qpos = ent.index.get_indexer(b["s1_entity_id"])
    cpos = ent.index.get_indexer(b["candidate_entity_id"])
    missing = int((qpos < 0).sum() + (cpos < 0).sum())
    assert missing == 0, f"{missing} bucket records are missing from processed/"

    def pair(col):
        v = ent[col].to_numpy()
        return v[qpos], v[cpos]

    f = {}
    by_s1 = b.groupby("s1_entity_id", sort=False)
    f["rank"] = b["rank"].to_numpy(np.float32)
    f["score"], f["cosine"] = b["score"].to_numpy(np.float32), b["cosine"].to_numpy(np.float32)
    f["s1_best_cosine"] = by_s1["cosine"].transform("max").to_numpy(np.float32)
    f["gap_to_best_score"] = (by_s1["score"].transform("max") - b["score"]).to_numpy(np.float32)
    f["gap_to_next_score"] = (b["score"] - by_s1["score"].shift(-1)).to_numpy(np.float32)
    f["s1_top5_cosine"] = b["s1_entity_id"].map(b[b["rank"] < 5].groupby("s1_entity_id")["cosine"].mean()).to_numpy(np.float32)
    for c in ("cand_n_s1", "cand_rank", "cand_margin"):
        f[c] = b[c].to_numpy(np.float32)
    x, y = pair("name_norm")
    f["name_ratio"], f["name_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    f["name_token_sort"], f["name_partial"] = sim(fuzz.token_sort_ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["s1_name_len"] = np.fromiter((len(v) for v in x), np.float32, len(x))
    f["cand_name_len"] = np.fromiter((len(v) for v in y), np.float32, len(y))
    x, y = pair("core_name")
    f["core_ratio"], f["core_jaro_winkler"] = sim(fuzz.ratio, x, y), sim(JaroWinkler.normalized_similarity, x, y)
    x, y = pair("name_lat")
    f["latin_ratio"], f["latin_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    x, y = pair("raw_name")
    f["raw_name_token_set"] = sim(fuzz.token_set_ratio, x, y)
    x, y = pair("compact")
    f["compact_ratio"], f["compact_partial"] = sim(fuzz.ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["phonetic_same"] = same(*pair("phonetic_code"))
    f["legal_form_same"] = same(*pair("legal_form"))
    x, y = pair("addr_norm")
    f["addr_ratio"], f["addr_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    x, y = pair("raw_addr")
    f["raw_addr_token_set"], f["raw_addr_partial"] = sim(fuzz.token_set_ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["cand_addr_len"] = np.fromiter((len(v) for v in y), np.float32, len(y))
    f["street_jaccard"], f["street_cover"] = set_overlap(*pair("street_set"))
    f["numbers_jaccard"], f["numbers_cover"] = set_overlap(*pair("numbers"))
    f["postcode_same"] = same(*pair("postcode"))
    f["house_no_same"] = same(*pair("house_no"))
    f["city_same"] = same(*pair("city"))
    f["state_same"] = same(*pair("state_code"))
    f["cand_is_s3"] = (b["candidate_source"].to_numpy() == "S3").astype(np.float32)
    for tag in TAGS:
        f[f"cand_{tag[4:]}"] = pair(tag)[1].astype(np.float32)
    if FEATURE_SET == "base":
        return np.column_stack([f[c] for c in FEATURES]).astype(np.float32)
    for kind in ("name", "addr"):
        w = weighted_overlap(*pair(f"{kind}_words"), stats[f"{kind}_idf"], stats["unseen_idf"])
        for j, c in enumerate(("jaccard", "cover", "miss_s1", "miss_cand")):
            f[f"{kind}_idf_{c}"] = w[j]
            f[f"{kind}_idf_{c}_fuzzy"] = w[4 + j]
    x, y = pair("street_name")
    f["street_name_token_set"], f["street_name_ratio"] = sim(fuzz.token_set_ratio, x, y), sim(fuzz.ratio, x, y)
    f["house_number_same"] = same(*pair("house_number"))
    x, y = pair("name_key")
    freq, per = stats["name_freq"], 100_000 / stats["n"]             # S1 with this name per 100,000 S1 of the country
    f["s1_name_freq"] = np.log1p(per * np.fromiter((freq.get(v, 0) for v in x), np.float32, len(x)))
    f["cand_name_freq"] = np.log1p(per * np.fromiter((freq.get(v, 0) for v in y), np.float32, len(y)))
    for c in SOURCE_FEATURES:                                        # empty when 02 ran without extra sources
        f[c] = b[c].to_numpy(np.float32) if c in b.columns else np.full(len(b), np.nan, np.float32)
    # candidate vs candidate: each candidate compared with the other candidates of the same S1
    codes = pd.factorize(b["s1_entity_id"].to_numpy())[0]
    for c in REL_SIMS:
        v = pd.Series(f[c])
        grp = v.groupby(codes)
        f[f"{c}_gap_best"] = (grp.transform("max") - v).to_numpy(np.float32)
        f[f"{c}_rank"] = (grp.rank(method="min", ascending=False) - 1).to_numpy(np.float32)
    name_ts, addr_ts = pd.Series(f["name_token_set"]), pd.Series(f["addr_token_set"])
    f["n_name_twins"] = (name_ts >= 90).groupby(codes).transform("sum").to_numpy(np.float32)
    f["n_addr_twins"] = (addr_ts >= 90).groupby(codes).transform("sum").to_numpy(np.float32)
    better = np.zeros(len(b), np.float32)            # candidates with a name at least as close and a clearly closer address
    nv, av = name_ts.fillna(-1).to_numpy(), addr_ts.fillna(-1).to_numpy()
    starts = np.r_[0, np.flatnonzero(codes[1:] != codes[:-1]) + 1, len(codes)]
    for a, e in zip(starts[:-1], starts[1:]):
        n_, a_ = nv[a:e], av[a:e]
        better[a:e] = ((n_[None, :] >= n_[:, None]) & (a_[None, :] >= a_[:, None] + 10)).sum(axis=1)
    f["better_addr_same_name"] = better
    return np.column_stack([f[c] for c in FEATURES]).astype(np.float32)


def featurize(b, ent, stats):
    """make_features over chunks of whole S1 buckets."""
    return np.vstack([make_features(b.iloc[s:e], ent, stats) for s, e in chunk_bounds(b)])


print(f"{len(FEATURES)} features")

## Ground truth
The true S1 of every train S2/S3 record, and the number of true matches of every train S1
(including matches that are not in its bucket, so they count as misses).

In [ ]:
gt = pd.read_csv(GT_FILE, sep="\t", dtype=object, keep_default_na=False, na_filter=False, quoting=csv.QUOTE_NONE)
GT_LIST = gt.set_index("source1_entity_id")["matched_entity_ids"]
pairs = gt[gt["matched_entity_ids"] != ""]
pairs = pairs.assign(entity_id=pairs["matched_entity_ids"].str.split(",")).explode("entity_id")
OWNER = pd.Series(pairs["source1_entity_id"].to_numpy(), index=pairs["entity_id"].to_numpy())
del gt, pairs
print(f"{len(GT_LIST):,} train S1, {int((GT_LIST == '').sum()):,} singletons, {len(OWNER):,} true pairs")

## Training pairs from the full-size train buckets
For each country: `TRAIN_S1_PER_COUNTRY` random train S1 with their buckets. The candidate-side
features were computed over every S1 of the country before the subset was taken.

In [ ]:
rng = np.random.default_rng(SEED)
X_parts, L_parts = [], []
for country in TRAIN_COUNTRIES:
    t0 = time.time()
    b = read_buckets("train", country)
    s1_all = b["s1_entity_id"].unique()
    chosen = rng.choice(s1_all, min(TRAIN_S1_PER_COUNTRY, len(s1_all)), replace=False)
    b = b[b["s1_entity_id"].isin(set(chosen))].reset_index(drop=True)
    ent = load_records("train", country, set(b["s1_entity_id"]) | set(b["candidate_entity_id"]))
    X_parts.append(featurize(b, ent, country_stats("train", country)))
    part = b[["s1_entity_id", "candidate_entity_id", "rank", "score", "cosine"]].copy()
    part["country"] = country
    part["is_true"] = part["candidate_entity_id"].map(OWNER).to_numpy() == part["s1_entity_id"].to_numpy()
    L_parts.append(part)
    print(f"train {country}: {len(s1_all):,} S1 in buckets, {len(chosen):,} used, {len(part):,} pairs "
          f"({int(part['is_true'].sum()):,} true), {time.time() - t0:.0f}s")
    del b, ent
    gc.collect()
X = np.vstack(X_parts)
L = pd.concat(L_parts, ignore_index=True)
y = L["is_true"].to_numpy()
del X_parts, L_parts
gc.collect()

# scored S1 = every training S1; true matches from the ground truth (bucket misses count)
EVAL_IDS = L["s1_entity_id"].unique()
TRUE_OF = {s: set(v.split(",")) if v else set() for s, v in GT_LIST.reindex(EVAL_IDS).fillna("").items()}
N_TRUE = pd.Series({s: len(v) for s, v in TRUE_OF.items()}, name="n_true").reindex(EVAL_IDS)
EVAL_COUNTRY = pd.Series(L.drop_duplicates("s1_entity_id").set_index("s1_entity_id")["country"]).reindex(EVAL_IDS).to_numpy()
L["s1_pos"] = L["s1_entity_id"].map(pd.Series(np.arange(len(EVAL_IDS)), index=EVAL_IDS)).to_numpy()
print(f"{len(L):,} training pairs, {len(EVAL_IDS):,} S1 ({int((N_TRUE == 0).sum()):,} singletons), "
      f"{X.nbytes / 1e9:.1f} GB of features, {time.time() - T0:.0f}s so far")

## Cross-validation of every model
The training S1 are split into the same folds as in 03. Each fold is predicted by models trained on the others.
For the network, the held-out log-loss is printed after every pass. If it rises in the last passes, lower
`NN_EPOCHS`.

In [ ]:
fold_of = pd.Series(rng.permutation(np.arange(len(EVAL_IDS)) % N_FOLDS), index=EVAL_IDS)
L["fold"] = L["s1_entity_id"].map(fold_of).to_numpy()

if "nn" in MODELS:
    t_nn = time.time()
    NN_Q, NN_EMPTY = nn_fit_transform(X)
    XN = nn_matrix(X)
    YN = torch.from_numpy(y.astype(np.float32)).to(NN_DEVICE)
    print(f"network inputs: {XN.shape[1]} columns ({len(NN_EMPTY)} 'is empty' flags), "
          f"{XN.numel() * XN.element_size() / 1e9:.1f} GB on {NN_DEVICE}, {time.time() - t_nn:.0f}s")

OOF = {m: np.zeros(len(L), np.float32) for m in MODELS}
ROUNDS_CV = {m: [] for m in MODELS}
FIT_LOG = []
for fold in range(N_FOLDS):
    tr, va = L["fold"].to_numpy() != fold, L["fold"].to_numpy() == fold
    for m in MODELS:
        t_m = time.time()
        model, rounds = fit_model(m, tr, va)
        OOF[m][va] = predict_model(m, model, rounds, rows=va)
        ROUNDS_CV[m].append(int(rounds))
        FIT_LOG.append({"fold": fold, "model": m, "rounds": int(rounds), "held-out log-loss": round(logloss(y[va], OOF[m][va]), 5),
                        "seconds": round(time.time() - t_m)})
        print(f"fold {fold} {m}: {int(tr.sum()):,} train / {int(va.sum()):,} held-out pairs, {rounds} rounds, "
              f"held-out log-loss {FIT_LOG[-1]['held-out log-loss']:.4f}, {FIT_LOG[-1]['seconds']}s", flush=True)
        del model
        free_gpu()
pd.DataFrame(FIT_LOG).pivot(index="model", columns="fold", values="held-out log-loss").round(4)

## Model comparison and blends
Each model and blend gets its own tuned cutoff t and the decision rule of 03 (keep p ≥ t, one owner per S2/S3).

**Columns**
* `gain vs lgbm`: the difference in macro F0.5 from LightGBM alone. `± 2 SE` is two standard errors of that
  difference over S1 (paired: both are scored on the same S1). A gain smaller than its ± can't be told apart from
  noise.
* `S1 scored differently`: how many S1 get a different score than with LightGBM. Few means the models make the
  same decisions, so blending them can't change much.

The second table is the correlation of the models' held-out log-odds: models that agree this closely can't fix
each other's mistakes.

In [ ]:
def per_s1_f05(df, keep):
    """Challenge F0.5 of every scored S1 (in EVAL_IDS order) when the rows of df selected by keep are the predictions."""
    pos = df["s1_pos"].to_numpy()
    n_pred = np.bincount(pos[keep], minlength=len(N_TRUE))
    tp = np.bincount(pos[keep], weights=df["is_true"].to_numpy()[keep].astype(float), minlength=len(N_TRUE))
    n_true = N_TRUE.to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        p, r = tp / n_pred, tp / n_true
        f = np.where(tp > 0, 1.25 * p * r / (0.25 * p + r), 0.0)
    return np.where(n_true == 0, (n_pred == 0).astype(float), f)


def score(df, keep):
    """Challenge macro F0.5 over the scored S1 when the rows of df selected by keep are the predictions."""
    f, n_true = per_s1_f05(df, keep), N_TRUE.to_numpy()
    out = {"macro F0.5": f.mean(), "S1 with matches": f[n_true > 0].mean(), "singletons": f[n_true == 0].mean()}
    out.update({f"macro F0.5 {c}": f[EVAL_COUNTRY == c].mean() for c in sorted(set(EVAL_COUNTRY))})
    return out


def one_owner(df, p):
    """True for the rows where this S1 gives the candidate its highest p among all S1 in df."""
    order = np.argsort(-p, kind="stable")
    first = ~pd.Series(df["candidate_entity_id"].to_numpy()[order]).duplicated().to_numpy()
    owner = np.zeros(len(df), bool)
    owner[order[first]] = True
    return owner


def tune(df, p, extra, grid):
    """Best threshold on grid for keep = (p >= t) & extra, and its scores."""
    return max(((t, score(df, (p >= t) & extra)) for t in grid), key=lambda r: r[1]["macro F0.5"])


def option_prob(name, probs):
    """Probability of one option (a model, `mean` or `stack`) from the models' probabilities {model: array}."""
    if name == "mean":
        return np.mean([probs[m] for m in MODELS], axis=0).astype(np.float32)
    if name == "stack":
        z = STACK_B + sum(STACK_W[m] * logit(probs[m]) for m in MODELS)
        return (1 / (1 + np.exp(-z))).astype(np.float32)
    return probs[name]


GRID = np.round(np.arange(0.02, 0.99, 0.01), 2)
BASE = "lgbm" if "lgbm" in MODELS else MODELS[0]
OPTIONS = {m: OOF[m] for m in MODELS}
STACK_W, STACK_B = {}, 0.0
if len(MODELS) > 1:
    OPTIONS["mean"] = option_prob("mean", OOF)
    Z = np.column_stack([logit(OOF[m]) for m in MODELS])
    folds, stack_rng = L["fold"].to_numpy(), np.random.default_rng(SEED)

    def fit_stack(mask):
        """Logistic regression on the models' log-odds, on at most 2M of the rows selected by mask."""
        rows = np.flatnonzero(mask)
        if len(rows) > 2_000_000:
            rows = stack_rng.choice(rows, 2_000_000, replace=False)
        return LogisticRegression(C=1.0, max_iter=1000).fit(Z[rows], y[rows])

    stack_oof = np.zeros(len(L), np.float32)
    for fold in range(N_FOLDS):
        stack_oof[folds == fold] = fit_stack(folds != fold).predict_proba(Z[folds == fold])[:, 1]
    OPTIONS["stack"] = stack_oof
    STACK = fit_stack(np.ones(len(L), bool))
    STACK_W, STACK_B = dict(zip(MODELS, STACK.coef_[0].tolist())), float(STACK.intercept_[0])
    del Z
    print("stack: weights on the log-odds", {m: round(w, 3) for m, w in STACK_W.items()}, f"intercept {STACK_B:.3f}")

rows, F_OF, T_OF = [], {}, {}
for name, p_opt in OPTIONS.items():
    owner = one_owner(L, p_opt)
    T_OF[name], s = tune(L, p_opt, owner, GRID)
    F_OF[name] = per_s1_f05(L, (p_opt >= T_OF[name]) & owner)
    rows.append({"option": name, "t": T_OF[name], **s, "held-out log-loss": logloss(y, p_opt)})
COMPARISON = pd.DataFrame(rows).set_index("option")
diff = {name: F_OF[name] - F_OF[BASE] for name in OPTIONS}
COMPARISON.insert(2, f"gain vs {BASE}", [diff[n].mean() for n in COMPARISON.index])
COMPARISON.insert(3, "± 2 SE", [2 * diff[n].std(ddof=1) / np.sqrt(len(diff[n])) for n in COMPARISON.index])
COMPARISON.insert(4, "S1 scored differently", [int((np.abs(diff[n]) > 1e-12).sum()) for n in COMPARISON.index])
COMPARISON.to_csv(OUT_DIR / "model" / "model_comparison.csv")

BEST = COMPARISON["macro F0.5"].idxmax()
if ENSEMBLE != "auto":
    CHOSEN = ENSEMBLE
elif BEST != BASE and COMPARISON.loc[BEST, f"gain vs {BASE}"] > COMPARISON.loc[BEST, "± 2 SE"]:
    CHOSEN = BEST
else:
    CHOSEN = BASE
    print(f"best option: {BEST}, but its gain over {BASE} is within ± 2 SE (noise), so {BASE} is used")
assert CHOSEN in OPTIONS, f"ENSEMBLE={ENSEMBLE}: use auto or one of {list(OPTIONS)}"
T_BEST = T_OF[CHOSEN]
L["p"] = OPTIONS[CHOSEN]
for m in MODELS:
    L[f"p_{m}"] = OOF[m]
p, cos = L["p"].to_numpy(), L["cosine"].to_numpy()
owner_p, owner_cos = one_owner(L, p), one_owner(L, cos)
s_best = score(L, (p >= T_BEST) & owner_p)
t_cos, s_cos = tune(L, cos, owner_cos, np.round(np.arange(0.80, 0.995, 0.005), 3))
METRIC = pd.DataFrame([{"method": "predict nothing", "t": None, **score(L, np.zeros(len(L), bool))},
                       {"method": "cosine ≥ t + one owner (notebook 02 rule)", "t": t_cos, **s_cos},
                       *[{"method": f"{n} p ≥ t + one owner" + (" (used for the submission)" if n == CHOSEN else ""),
                          "t": T_OF[n], **score(L, (OPTIONS[n] >= T_OF[n]) & one_owner(L, OPTIONS[n]))} for n in OPTIONS],
                       {"method": "ceiling: perfect matcher on the buckets", "t": None, **score(L, L["is_true"].to_numpy())}])
METRIC.to_csv(OUT_DIR / "model" / "metric_f05.csv", index=False)
L["kept"] = (p >= T_BEST) & owner_p
L[["s1_entity_id", "candidate_entity_id", "country", "rank", "cosine", "fold", "p", *[f"p_{m}" for m in MODELS],
   "is_true", "kept"]].to_parquet(OUT_DIR / "model" / "oof_pairs.parquet", index=False)

kept = L[L["kept"]]
reference = macro_f05(kept.groupby("s1_entity_id")["candidate_entity_id"].apply(list).to_dict(), TRUE_OF, EVAL_IDS)
assert abs(reference - s_best["macro F0.5"]) < 1e-9, (reference, s_best["macro F0.5"])
print(f"chosen: {CHOSEN} (ENSEMBLE={ENSEMBLE}), threshold t = {T_BEST}; vectorized score matches the reference metric")
if len(MODELS) > 1:
    display(pd.DataFrame(np.corrcoef([logit(OOF[m]) for m in MODELS]), index=MODELS, columns=MODELS).round(3))
COMPARISON.round(4)

### Score at other thresholds (chosen option)
A flat curve around t means the choice is not fragile.

In [ ]:
pd.DataFrame([{"t": t, **score(L, (p >= t) & owner_p)} for t in np.round(np.arange(0.1, 0.95, 0.05), 2)]) \
    .set_index("t")[["macro F0.5", "S1 with matches", "singletons"]].round(4)

## Transfer between countries (stand-in for France)
Each model is trained on one country and scores the other, with its own cutoff from the comparison above. The
blends reuse the stack weights from cross-validation, which saw both countries, so their rows are slightly
optimistic. A model that loses much more here than the others transfers worse to an unseen country, which matters
for France.

In [ ]:
ROUNDS = {m: NN_EPOCHS if m == "nn" else max(50, int(np.mean(ROUNDS_CV[m]) * 1.1)) for m in MODELS}
USED = MODELS if CHOSEN in ("mean", "stack") else [CHOSEN]
TRANSFER = []
if CROSS_COUNTRY_CHECK and len(TRAIN_COUNTRIES) > 1:
    for test_c in TRAIN_COUNTRIES:
        t_c = time.time()
        tr, te = L["country"].to_numpy() != test_c, L["country"].to_numpy() == test_c
        sub, probs = L[te], {}
        for m in MODELS:
            model, rounds = fit_model(m, tr, rounds=ROUNDS[m])
            probs[m] = predict_model(m, model, rounds, rows=te)
            del model
            free_gpu()
        for name in OPTIONS:
            p_te = option_prob(name, probs)
            s = score(sub, (p_te >= T_OF[name]) & one_owner(sub, p_te))[f"macro F0.5 {test_c}"]
            TRANSFER.append({"option": name, "trained on": " + ".join(c for c in TRAIN_COUNTRIES if c != test_c),
                             "scored on": test_c, "macro F0.5": round(s, 4),
                             "cross-validation (all countries)": round(COMPARISON.loc[name, f"macro F0.5 {test_c}"], 4)})
        print(f"transfer to {test_c}: {time.time() - t_c:.0f}s", flush=True)
        del sub, probs
        gc.collect()
pd.DataFrame(TRANSFER).pivot(index="option", columns="scored on", values=["macro F0.5", "cross-validation (all countries)"]) \
    if TRANSFER else "transfer skipped"

## Final models on all training pairs
Only the models the chosen option uses.

In [ ]:
t_final = time.time()
FINAL = {}
for m in USED:
    t_m = time.time()
    FINAL[m] = fit_model(m, rounds=ROUNDS[m])
    print(f"final {m}: {ROUNDS[m]} rounds, {time.time() - t_m:.0f}s", flush=True)
    free_gpu()
MODEL_DIR = OUT_DIR / "model"
if "lgbm" in FINAL:
    FINAL["lgbm"][0].save_model(str(MODEL_DIR / "lgbm_matcher.txt"))
if "xgb" in FINAL:
    FINAL["xgb"][0].save_model(str(MODEL_DIR / "xgb_matcher.json"))
if "cat" in FINAL:
    FINAL["cat"][0].save_model(str(MODEL_DIR / "cat_matcher.cbm"))
if "nn" in FINAL:
    torch.save({"state_dict": FINAL["nn"][0].state_dict(), "hidden": NN_HIDDEN, "quantiles": NN_Q,
                "empty_columns": NN_EMPTY.tolist(), "features": FEATURES}, MODEL_DIR / "nn_matcher.pt")
IMPORTANCE = None
if "lgbm" in FINAL:
    IMPORTANCE = pd.DataFrame({"feature": FEATURES, "gain": FINAL["lgbm"][0].feature_importance("gain"),
                               "splits": FINAL["lgbm"][0].feature_importance("split")}).sort_values("gain", ascending=False)
    IMPORTANCE["gain %"] = (100 * IMPORTANCE["gain"] / IMPORTANCE["gain"].sum()).round(2)
    IMPORTANCE.to_csv(MODEL_DIR / "feature_importance.csv", index=False)
TRAIN_RATES = L.groupby("country").apply(lambda g: pd.Series({
    "S1 with a predicted match (cv)": g.loc[g["kept"], "s1_entity_id"].nunique() / g["s1_entity_id"].nunique(),
    "S1 with a true match": float((N_TRUE.reindex(g["s1_entity_id"].unique()) > 0).mean()),
    "matches per matched S1 (cv)": g["kept"].sum() / max(g.loc[g["kept"], "s1_entity_id"].nunique(), 1)}))
del X, XN, YN
free_gpu()
print(f"final models ({', '.join(USED)}) in {time.time() - t_final:.0f}s")
IMPORTANCE[["feature", "gain %", "splits"]].head(15) if IMPORTANCE is not None else None

## Predict the test buckets
Per country: features in chunks, the probability of every model the chosen option uses for every pair, the blend, one owner per S2/S3, keep p ≥ t. The records
(names, addresses, word sets) are loaded for one of `PRED_PARTS` S1 groups at a time: loading a whole country's
made submission 2's run fail with an out-of-memory error. Same probabilities as in one piece (a group never
splits an S1's bucket, and the candidate-side statistics are computed on the whole bucket first).
The candidates of an S1 are exactly the pairs scored here, in rank order; its matches are the kept
ones, most likely first.

**Check France here.** France has no labels, so its score cannot be measured. Compare its
"S1 with a predicted match" and "matches per matched S1" with India and the US, and with the
training rates printed above them (about 94% and 3.5). Much higher values for France mean false
matches: in a local test on a thin 1% sample, France matched lookalikes (same city, house
number, a generic word and legal form, different street) about 3x as often as India/US. If that
shows up here, set a stricter cutoff, e.g. `COUNTRY_THRESHOLDS = {"France": 0.9}`, and re-run.

In [ ]:
def part_bounds(b, parts):
    """(start, end) row ranges of about len(b) / parts rows that never split an S1's bucket."""
    s1 = b["s1_entity_id"].to_numpy()
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    targets = np.linspace(0, len(b), parts + 1)[1:-1]
    cuts = starts[np.minimum(np.searchsorted(starts, targets), len(starts) - 1)]
    cuts = np.unique(np.r_[0, cuts, len(b)])
    return list(zip(cuts[:-1], cuts[1:]))


def join_by_s1(s1, ids):
    """{s1: 'id,id,...'} for rows grouped by S1 (order kept, duplicates removed)."""
    if not len(s1):
        return {}
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    ends = np.r_[starts[1:], len(s1)]
    return {s1[a]: ",".join(dict.fromkeys(ids[a:e])) for a, e in zip(starts, ends)}


CANDIDATES, MATCHES, TEST_STATS = {}, {}, []
for country in TEST_COUNTRIES:
    t0 = time.time()
    b = read_buckets("test", country)
    stats = country_stats("test", country)
    probs = {m: np.empty(len(b), np.float32) for m in USED}
    for k, (lo, hi) in enumerate(part_bounds(b, PRED_PARTS)):
        part = b.iloc[lo:hi]
        ent = load_records("test", country, set(part["s1_entity_id"]) | set(part["candidate_entity_id"]))
        for s, e in chunk_bounds(part):
            features = make_features(part.iloc[s:e], ent, stats)
            for m in USED:
                probs[m][lo + s:lo + e] = predict_model(m, *FINAL[m], x=features)
            del features
        del ent, part
        free_gpu()
        print(f"  {country} group {k + 1}/{PRED_PARTS}: rows {lo:,}-{hi:,}, {time.time() - t0:.0f}s", flush=True)
    prob = option_prob(CHOSEN, probs)
    t_country = COUNTRY_THRESHOLDS.get(country, T_BEST)
    keep = (prob >= t_country) & one_owner(b, prob)
    CANDIDATES.update(join_by_s1(b["s1_entity_id"].to_numpy(), b["candidate_entity_id"].to_numpy()))
    kept = pd.DataFrame({"s1": b["s1_entity_id"].to_numpy()[keep], "cand": b["candidate_entity_id"].to_numpy()[keep],
                         "p": prob[keep]}).sort_values(["s1", "p"], ascending=[True, False])
    MATCHES.update(join_by_s1(kept["s1"].to_numpy(), kept["cand"].to_numpy()))
    pd.DataFrame({"s1_entity_id": b["s1_entity_id"], "candidate_entity_id": b["candidate_entity_id"],
                  "rank": b["rank"], "p": prob, **({f"p_{m}": probs[m] for m in USED} if len(USED) > 1 else {})}) \
        .to_parquet(OUT_DIR / "test_probabilities" / f"{country}.parquet", index=False)
    n_s1 = b["s1_entity_id"].nunique()
    TEST_STATS.append({"country": country, "t": t_country, "S1 with buckets": n_s1, "pairs scored": len(b),
                       "S1 with a predicted match": round(kept["s1"].nunique() / n_s1, 4),
                       "matches per matched S1": round(len(kept) / max(kept["s1"].nunique(), 1), 2),
                       "seconds": round(time.time() - t0)})
    print(f"test {country}: {n_s1:,} S1, {len(b):,} pairs, {len(kept):,} matches, {time.time() - t0:.0f}s")
    del b, prob, probs, keep, kept, stats
    gc.collect()
print("training data, for comparison:")
display(TRAIN_RATES.round(3))
pd.DataFrame(TEST_STATS)

## Write the submission files
Every S1 in `test_source1.tsv` gets exactly one row, in file order, with an empty list when it
has no candidates or no match. The files are tab-separated, UTF-8 and use `\n` line endings,
with the headers from the problem statement.

In [ ]:
def read_ids(path):
    """First-column IDs of a TSV, read exactly as validate_submission.py reads them (file order, no duplicates)."""
    with open(path, encoding="utf-8") as f:
        next(f, None)
        return list(dict.fromkeys(line.split("\t", 1)[0].strip() for line in f if line.strip()))


def write_tsv(path, header, values, s1_ids):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s in s1_ids:
            f.write(f"{s}\t{values.get(s, '')}\n")


if PREDICT_TEST:
    TEST_S1 = read_ids(TEST_S1_FILE)
    MATCHING_FILE = OUT_DIR / "output" / "matching_results.tsv"
    CANDIDATE_FILE = OUT_DIR / "output" / "candidate_pairs.tsv"
    write_tsv(MATCHING_FILE, ["source1_entity_id", "matched_entity_ids"], MATCHES, TEST_S1)
    write_tsv(CANDIDATE_FILE, ["source1_entity_id", "candidate_entity_ids"], CANDIDATES, TEST_S1)
    not_in_test = set(CANDIDATES) - set(TEST_S1)
    print(f"{len(TEST_S1):,} test S1 written | with candidates: {sum(s in CANDIDATES for s in TEST_S1):,} | "
          f"with matches: {sum(s in MATCHES for s in TEST_S1):,} | bucket S1 not in test_source1.tsv: {len(not_in_test)}")
    print(f"{MATCHING_FILE}: {MATCHING_FILE.stat().st_size / 1e6:.0f} MB | {CANDIDATE_FILE}: {CANDIDATE_FILE.stat().st_size / 1e6:.0f} MB")
else:
    print("PREDICT_TEST=false: no submission files written")

CONFIG = {"features": FEATURES, "models": MODELS, "ensemble": CHOSEN, "models_used": USED,
          "stack_weights": STACK_W, "stack_intercept": STACK_B,
          "params": {"lgbm": PARAMS, "xgb": XGB_PARAMS, "cat": CAT_PARAMS,
                     "nn": {"hidden": NN_HIDDEN, "epochs": NN_EPOCHS, "batch": NN_BATCH, "lr": NN_LR}},
          "rounds": ROUNDS, "fold_rounds": ROUNDS_CV, "fit_log": FIT_LOG,
          "threshold": T_BEST, "thresholds_per_option": T_OF, "country_thresholds": COUNTRY_THRESHOLDS, "one_owner": True,
          "candidates_per_s1": CANDIDATES_PER_S1, "train_s1_per_country": TRAIN_S1_PER_COUNTRY,
          "cv_scores": METRIC.to_dict("records"), "model_comparison": COMPARISON.reset_index().to_dict("records"),
          "transfer": TRANSFER, "test": TEST_STATS, "buckets": str(BUCKET_DIR), "processed": str(PROCESSED_DIR),
          "minutes": round((time.time() - T0) / 60, 1)}
_ = (OUT_DIR / "model" / "matcher_config.json").write_text(json.dumps(CONFIG, indent=1, default=str), encoding="utf-8")

## Check the files against the submission rules
The same rules as `validate_submission.py`, including its optional ID-existence check: header,
one row per test S1 (no missing, extra or duplicate rows), no duplicate IDs in a list, only
S2-/S3- IDs, every ID exists in the test S2/S3 files, and matches ⊆ candidates. Then the official
validator runs too, if it is attached as an input (it is stdlib-only).

In [ ]:
def check_file(path, header, required, valid_ids):
    """Rule violations of one submission file, and its {s1: set(ids)} mapping."""
    problems, mapping, rows = [], {}, []
    with open(path, encoding="utf-8") as f:
        if f.readline().rstrip("\n").split("\t") != header:
            problems.append(f"header is not {header}")
        for line in f:
            s1, tab, rest = line.partition("\t")
            if not tab:
                problems.append(f"row without a tab: {line[:60]!r}")
                continue
            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            rows.append(s1)
            if len(ids) != len(set(ids)):
                problems.append(f"duplicate ID in the list of {s1}")
            mapping[s1] = set(ids)
    bad_prefix = {i for v in mapping.values() for i in v if not i.startswith(("S2-", "S3-"))}
    unknown = {i for v in mapping.values() for i in v if i not in valid_ids}
    for label, found in [("duplicate S1 rows", len(rows) - len(set(rows))), ("missing test S1", len(required - set(rows))),
                         ("S1 not in the test set", len(set(rows) - required)), ("IDs without S2-/S3- prefix", len(bad_prefix)),
                         ("IDs not in test S2/S3", len(unknown))]:
        if found:
            problems.append(f"{found:,} {label}")
    return problems, mapping


if PREDICT_TEST:
    required = set(TEST_S1)
    valid_ids = set(read_ids(TEST_DIR / "test_source2.tsv")) | set(read_ids(TEST_DIR / "test_source3.tsv"))
    problems_m, matched = check_file(MATCHING_FILE, ["source1_entity_id", "matched_entity_ids"], required, valid_ids)
    problems_c, candidates = check_file(CANDIDATE_FILE, ["source1_entity_id", "candidate_entity_ids"], required, valid_ids)
    not_subset = sum(1 for s, v in matched.items() if v - candidates.get(s, set()))
    print("matching_results.tsv:", problems_m or "OK")
    print("candidate_pairs.tsv:", problems_c or "OK")
    print("matches that are not candidates:", not_subset or "none")
    assert not problems_m and not problems_c and not not_subset, "fix the problems above before submitting"
    del valid_ids

    if VALIDATOR is not None:
        result = subprocess.run([sys.executable, str(VALIDATOR), "--matching", str(MATCHING_FILE), "--candidate",
                                 str(CANDIDATE_FILE), "--test-dir", str(TEST_DIR), "--check-ids"],
                                capture_output=True, text=True, encoding="utf-8", errors="replace")
        print(result.stdout[-4000:], result.stderr[-2000:])
    else:
        print("validate_submission.py not attached: run it locally on the downloaded files (see the notebook text)")
print(f"done in {(time.time() - T0) / 60:.1f} min")